## Detect Hand Gesture

In [2]:
"""
Hand Gesture Detection using MediaPipe Tasks API
--------------------------------------------------
Works with mediapipe >= 0.10.x AND the new mediapipe 1.0.1+ releases,
since it uses the modern `mediapipe.tasks` API instead of the removed
`mp.solutions` legacy API.

Detects: Fist, Open Palm, Thumbs Up, Thumbs Down, Peace Sign,
Pointing, OK Sign.

Requirements:
    pip install mediapipe opencv-python

First run will auto-download the hand landmark model (~10 MB) to the
same folder as this script.

Run:
    python hand_gesture_detection.py

Press 'q' to quit.
"""

import os
import math
import urllib.request

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

# ---------------------------------------------------------------------
# Model setup
# ---------------------------------------------------------------------
MODEL_PATH = os.path.join(os.getcwd(), "hand_landmarker.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/latest/hand_landmarker.task"
)


def ensure_model():
    if not os.path.exists(MODEL_PATH):
        print("Downloading hand_landmarker.task model...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
        print("Model downloaded to", MODEL_PATH)


# ---------------------------------------------------------------------
# Landmark indices (same layout as before)
# ---------------------------------------------------------------------
WRIST = 0
THUMB_TIP, THUMB_IP, THUMB_MCP = 4, 3, 2
INDEX_TIP, INDEX_PIP, INDEX_MCP = 8, 6, 5
MIDDLE_TIP, MIDDLE_PIP, MIDDLE_MCP = 12, 10, 9
RING_TIP, RING_PIP, RING_MCP = 16, 14, 13
PINKY_TIP, PINKY_PIP, PINKY_MCP = 20, 18, 17


def distance(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)


def finger_is_extended(landmarks, tip_idx, pip_idx, mcp_idx):
    wrist = landmarks[WRIST]
    tip_dist = distance(landmarks[tip_idx], wrist)
    pip_dist = distance(landmarks[pip_idx], wrist)
    return tip_dist > pip_dist * 1.1


def thumb_is_extended(landmarks, handedness_label):
    tip = landmarks[THUMB_TIP]
    mcp = landmarks[THUMB_MCP]
    if handedness_label == "Right":
        return tip.x < mcp.x
    else:
        return tip.x > mcp.x


def classify_gesture(landmarks, handedness_label):
    thumb_up = thumb_is_extended(landmarks, handedness_label)
    index_up = finger_is_extended(landmarks, INDEX_TIP, INDEX_PIP, INDEX_MCP)
    middle_up = finger_is_extended(landmarks, MIDDLE_TIP, MIDDLE_PIP, MIDDLE_MCP)
    ring_up = finger_is_extended(landmarks, RING_TIP, RING_PIP, RING_MCP)
    pinky_up = finger_is_extended(landmarks, PINKY_TIP, PINKY_PIP, PINKY_MCP)

    count_up = sum([thumb_up, index_up, middle_up, ring_up, pinky_up])

    ok_dist = distance(landmarks[THUMB_TIP], landmarks[INDEX_TIP])
    if ok_dist < 0.05 and middle_up and ring_up and pinky_up:
        return "OK Sign"

    if thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
        return "Thumbs Up" if landmarks[THUMB_TIP].y < landmarks[WRIST].y else "Thumbs Down"

    if count_up == 0:
        return "Fist"

    if index_up and middle_up and not ring_up and not pinky_up and not thumb_up:
        return "Peace / Victory"

    if index_up and not middle_up and not ring_up and not pinky_up and not thumb_up:
        return "Pointing"

    if count_up == 5:
        return "Open Palm"

    return "Unknown Gesture"


# ---------------------------------------------------------------------
# Drawing helper (manual, since mp.solutions.drawing_utils is gone)
# ---------------------------------------------------------------------
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),          # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),          # index
    (5, 9), (9, 10), (10, 11), (11, 12),     # middle
    (9, 13), (13, 14), (14, 15), (15, 16),   # ring
    (13, 17), (17, 18), (18, 19), (19, 20),  # pinky
    (0, 17),
]


def draw_landmarks(frame, landmarks):
    h, w, _ = frame.shape
    points = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]
    for start, end in HAND_CONNECTIONS:
        cv2.line(frame, points[start], points[end], (255, 255, 255), 2)
    for x, y in points:
        cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)


# ---------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------
def main():
    ensure_model()

    base_options = mp_tasks.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.6,
        min_tracking_confidence=0.6,
    )

    landmarker = mp_vision.HandLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(0)
    frame_timestamp_ms = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            continue

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        frame_timestamp_ms += 33  # approx one frame at ~30fps
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        if result.hand_landmarks:
            for landmarks, handedness in zip(result.hand_landmarks, result.handedness):
                label = handedness[0].category_name
                # Correct handedness because the camera frame is mirrored
                label = "Right" if label == "Left" else "Left"
                draw_landmarks(frame, landmarks)
                gesture = classify_gesture(landmarks, label)

                h, w, _ = frame.shape
                wrist_px = (int(landmarks[WRIST].x * w), int(landmarks[WRIST].y * h) + 30)
                cv2.putText(
                    frame, f"{label}: {gesture}", wrist_px,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA,
                )
        else:
            cv2.putText(
                frame, "No hand detected", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA,
            )

        cv2.imshow("Hand Gesture Detection", frame)
        if cv2.waitKey(5) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()


if __name__ == "__main__":
    main()

KeyboardInterrupt: 